<a href="https://colab.research.google.com/github/saivardhankundala2005-sys/langchain-primer/blob/main/Copy_of_ADK_Learning_tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Welcome to Your ADK Adventure - Tools & Memory! 🚀

Welcome, Agent Architect! This notebook is your guide to giving your AI agents two essential superpowers: custom tools and conversational memory.

By the end of this adventure, you will be able to:

- **Build a Foundational Agent**: Create a simple but effective AI agent from scratch using the Google Agent Development Kit (ADK).

- **Grant New Skills with Custom Tools**: Teach an agent to perform new tasks by connecting it to external APIs, like a real-time weather service.

- **Create a Team of Agents**: Assemble a multi-agent system where a primary agent can delegate specialized tasks to other agents.

- **Master Conversational Memory**: Understand the critical role of Sessions in enabling agents to remember previous interactions, handle feedback, and carry on a coherent conversation.


Let's get this adventure started!

## Author

HI, I'm Qingyue (Annie) Wang, a developer advocate and AI engineer at **Google**, passionate about helping developers build with AI and cloud technologies :)


If you have questions with this notebook, contact me on [LinkedIn](https://www.linkedin.com/in/anniewangtech/) , [X](https://twitter.com/anniewangtech) or email anniewangtech0510@Gmail.com


```
  (\__/)
  (•ㅅ•)
  /づ  📚      Enjoy learning AI Agents :)
```


-------------
### 🎁 🛑 Important Prerequisite: Setup Your Environment! 🛑 🎁
-----------------------------------------------------------------------------

👉 **Get Your API Key HERE**: https://codelabs.developers.google.com/onramp/instructions#1

 -----------------------------------------------------------------------------

```
 ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️
   /\_/\     /\_/\     /\_/\      /\_/\       /\_/\
  ( ^_^ )   ( -.- )   ( >_< )   ( =^.^= )    ( o_o )             
```


## Part 0: Setup & Authentication 🔑

First things first, let's get all our tools ready. This step installs the necessary libraries and securely configures your Google API key so your agents can access the power of Gemini.

In [ ]:
!pip install google-adk google-generativeai -q

# --- Import all necessary libraries ---
import os
import sys
import json
import asyncio
import random
import string
from uuid import uuid4
from typing import Any, List

import pandas as pd
import plotly.graph_objects as go
import vertexai
from google.colab import auth
from IPython.display import HTML, Markdown, display

# --- ADK, Agent, and Evaluation Components ---
from google.adk.agents import Agent
from google.adk.events import Event
from google.adk.runners import Runner
import google.adk as adk
from google.adk.tools import google_search
from google.adk.sessions import InMemorySessionService, Session
from google.genai import types
from google.genai.types import Content, Part


print("✅ All libraries are ready to go!")


✅ All libraries are ready to go!


/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


### Authenticate and Configure Your Project
To use Vertex AI, you need an active Google Cloud project. This section handles authenticating your environment and setting up the necessary project configurations.

In [ ]:
# ---  Authentication & Project Configuration ---

# Authenticate user in Colab
if "google.colab" in sys.modules:
    auth.authenticate_user()
    print("✅ Authenticated successfully.")

✅ Authenticated successfully.


In [ ]:
# @title Set Your Google Cloud Project Details
PROJECT_ID = "red-planet-497618-r3"             # @param {type:"string"}
LOCATION = "us-central1"               # @param {type:"string"}

# Set environment variables for the ADK and gcloud
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"

!gcloud services enable aiplatform.googleapis.com --project={PROJECT_ID}

print(f"\n✅ Vertex AI configured for project '{PROJECT_ID}' in '{LOCATION}'.")

Operation "operations/acat.p2-672578821759-e20f18e9-ae22-49e1-9e0c-7b74b80567a1" finished successfully.

✅ Vertex AI configured for project 'red-planet-497618-r3' in 'us-central1'.


---
## Part 1: Your First Agent - The Day Trip Genie 🧞

Meet your first creation! The `day_trip_agent` is a simple but powerful assistant. We're making it a little smarter by teaching it to understand **budget constraints**.

* **Agent**: The brain of the operation, defined by its instructions, tools, and the AI model it uses.
* **Session**: The conversation history. For this simple agent, it's just a container for a single request-response.
* **Runner**: The engine that connects the `Agent` and the `Session` to process your request and get a response.

```
+--------------------------------------------------+
|         Spontaneous Day Trip Agent 🤖            |
|--------------------------------------------------|
|  Model: gemini-2.5-flash                         |
|  Description:                                    |
|   Generates full-day trip itineraries based on   |
|   mood, interests, and budget                    |
|--------------------------------------------------|
|  🔧 Tools:                                       |
|   - Google Search                                |
|--------------------------------------------------|
|  🧠 Capabilities:                                |
|   - Budget Awareness (cheap / splurge)           |
|   - Mood Matching (adventurous, relaxing, etc.)  |
|   - Real-Time Info (hours, events)               |
|   - Morning / Afternoon / Evening plan           |
+--------------------------------------------------+

            ▲
            |
    +------------------+
    |   User Input     |
    |------------------|
    |  Mood            |
    |  Interests       |
    |  Budget          |
    +------------------+

            |
            ▼

+--------------------------------------------------+
|             Output: Markdown Itinerary           |
|--------------------------------------------------|
| - Time blocks (Morning / Afternoon / Evening)    |
| - Venue names with links and hours               |
| - Budget-matching activities                     |
+--------------------------------------------------+
```


In [ ]:
# --- Agent Definition ---

def create_day_trip_agent():
    """Create the Spontaneous Day Trip Generator agent"""
    return Agent(
        name="day_trip_agent",
        model="gemini-2.5-flash",
        description="Agent specialized in generating spontaneous full-day itineraries based on mood, interests, and budget.",
        instruction="""
        You are the "Spontaneous Day Trip" Generator 🚗 - a specialized AI assistant that creates engaging full-day itineraries.

        Your Mission:
        Transform a simple mood or interest into a complete day-trip adventure with real-time details, while respecting a budget.

        Guidelines:
        1. **Budget-Aware**: Pay close attention to budget hints like 'cheap', 'affordable', or 'splurge'. Use Google Search to find activities (free museums, parks, paid attractions) that match the user's budget.
        2. **Full-Day Structure**: Create morning, afternoon, and evening activities.
        3. **Real-Time Focus**: Search for current operating hours and special events.
        4. **Mood Matching**: Align suggestions with the requested mood (adventurous, relaxing, artsy, etc.).

        RETURN itinerary in MARKDOWN FORMAT with clear time blocks and specific venue names.
        """,
        tools=[google_search]
    )

day_trip_agent = create_day_trip_agent()
print(f"🧞 Agent '{day_trip_agent.name}' is created and ready for adventure!")

🧞 Agent 'day_trip_agent' is created and ready for adventure!


In [ ]:
# --- A Helper Function to Run Our Agents ---
# We'll use this function throughout the notebook to make running queries easy.

async def run_agent_query(agent: Agent, query: str, session: Session, user_id: str, is_router: bool = False):
    """Initializes a runner and executes a query for a given agent and session."""
    print(f"\n🚀 Running query for agent: '{agent.name}' in session: '{session.id}'...")

    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name
    )

    final_response = ""
    try:
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=Content(parts=[Part(text=query)], role="user")
        ):
            if not is_router:
                # Let's see what the agent is thinking!
                print(f"EVENT: {event}")
            if event.is_final_response():
                final_response = event.content.parts[0].text
    except Exception as e:
        final_response = f"An error occurred: {e}"

    if not is_router:
     print("\n" + "-"*50)
     print("✅ Final Response:")
     display(Markdown(final_response))
     print("-"*50 + "\n")

    return final_response

# --- Initialize our Session Service ---
# This one service will manage all the different sessions in our notebook.
session_service = InMemorySessionService()
my_user_id = "adk_adventurer_001"

In [ ]:
# --- Let's test the Day Trip Genie! ---

async def run_day_trip_genie():
    # Create a new, single-use session for this query
    day_trip_session = await session_service.create_session(
        app_name=day_trip_agent.name,
        user_id=my_user_id
    )

    # Note the new budget constraint in the query!
    query = "Plan a relaxing and artsy day trip near Sunnyvale, CA. Keep it affordable!"
    print(f"🗣️ User Query: '{query}'")

    await run_agent_query(day_trip_agent, query, day_trip_session, my_user_id)

await run_day_trip_genie()

🗣️ User Query: 'Plan a relaxing and artsy day trip near Sunnyvale, CA. Keep it affordable!'

🚀 Running query for agent: 'day_trip_agent' in session: '0b4bcb16-2310-4607-8281-6d5c3b7a087a'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Here's a relaxing and artsy day trip itinerary near Sunnyvale, CA, designed to be affordable! This trip focuses on the vibrant arts scene and serene environments of Palo Alto and nearby San Jose.

## 🎨🏞️ Relaxing & Artsy Day Trip near Sunnyvale (Affordable!)

**Mood:** Artsy, Relaxing
**Budget:** Affordable (prioritizing free activities and budget-friendly dining)
**Locations:** Palo Alto, Stanford, and San Jose

---

### ☀️ Morning (10:00 AM - 1:00 PM) - Palo Alto & Stanford University

Start your day with a journey into art and tranquility at Stanford University.

*   **10:00 AM - 11:00 AM: Stanford University Public Art Walk**
    Begin with a leisurely self-guided stroll through the beautiful Stanford cam

Here's a relaxing and artsy day trip itinerary near Sunnyvale, CA, designed to be affordable! This trip focuses on the vibrant arts scene and serene environments of Palo Alto and nearby San Jose.

## 🎨🏞️ Relaxing & Artsy Day Trip near Sunnyvale (Affordable!)

**Mood:** Artsy, Relaxing
**Budget:** Affordable (prioritizing free activities and budget-friendly dining)
**Locations:** Palo Alto, Stanford, and San Jose

---

### ☀️ Morning (10:00 AM - 1:00 PM) - Palo Alto & Stanford University

Start your day with a journey into art and tranquility at Stanford University.

*   **10:00 AM - 11:00 AM: Stanford University Public Art Walk**
    Begin with a leisurely self-guided stroll through the beautiful Stanford campus, which features over 80 works of outdoor public art accessible year-round. You can enjoy the architecture and various sculptures in a peaceful setting. Keep an eye out for the distinguished collection of 20th-century outdoor sculptures.
    *(Cost: Free; Parking at Stanford may have a fee, use ParkMobile app if visiting Mon-Fri, 8 AM - 4 PM, or look for free street parking further out.)*

*   **11:00 AM - 1:00 PM: Cantor Arts Center & Rodin Sculpture Garden**
    Head over to the Cantor Arts Center, which opens at 11:00 AM on Wednesdays. Admission is always free! Explore their diverse collections spanning 5,000 years of art, including a renowned Rodin collection and the peaceful B. Gerald Cantor Rodin Sculpture Garden.
    *(Cost: Free admission)*

### 🍽️ Lunch (1:00 PM - 2:00 PM) - Affordable Bites in Palo Alto

For an affordable and delicious lunch, explore some local favorites in Palo Alto.

*   **Option 1: Taqueria El Grullense M&G**
    Enjoy authentic Mexican cuisine with highly-rated burritos and tacos that are easy on the wallet.
    *(Cost: Affordable)*

*   **Option 2: Mediterranean Wraps**
    Another budget-friendly option, Mediterranean Wraps offers flavorful Mediterranean dishes in a welcoming atmosphere.
    *(Cost: Affordable)*

### 🖼️ Afternoon (2:30 PM - 5:00 PM) - Palo Alto Art Scene

Continue your artsy exploration in Palo Alto.

*   **2:30 PM - 4:30 PM: Palo Alto Art Center**
    This art center offers free admission and a welcoming environment to view various exhibitions by emerging and professional artists. It's a great spot to discover local and regional artwork.
    *(Cost: Free admission)*

*   **4:30 PM - 5:00 PM: Free Little Art Gallery of Palo Alto (Optional)**
    If you're interested in a unique, community-driven art experience, seek out the Free Little Art Gallery of Palo Alto. This miniature gallery encourages visitors to take or leave small pieces of art, offering "small doses of cheer, motivation, and humor."
    *(Cost: Free)*

### 🌆 Evening (5:30 PM - 8:00 PM) - San Jose Public Art & Dinner

Wind down your day with more public art and an affordable dinner in downtown San Jose.

*   **5:30 PM - 6:30 PM: Downtown San Jose Public Art Walk**
    Take a self-guided walk through downtown San Jose to discover an array of public art installations. Areas like Downtown Doors and the Art Box Project transform utility boxes and doors into canvases for local artists. You'll find impressive pieces around almost every corner.
    *(Cost: Free)*

*   **6:30 PM - 8:00 PM: Dinner at San Pedro Square Market**
    Conclude your day with dinner at San Pedro Square Market in downtown San Jose. This vibrant hub offers a diverse range of food stalls with affordable options, from pizza and burgers to various international cuisines. It's a lively spot to relax and enjoy a meal.
    *(Cost: Affordable)*

--------------------------------------------------



---
## Part 2: Supercharging Agents with Custom Tools 🛠️

So far, we've used the powerful built-in `GoogleSearch` tool. But the true power of agents comes from connecting them to your own logic and data sources.

This is where **custom tools** come in. Let's explore three patterns for giving your agent new skills, using real-world, practical examples.

### 2.1 The Simple `FunctionTool`: Calling a Real-Time Weather API

The most direct way to create a tool is by writing a Python function. This is perfect for synchronous tasks like fetching data from an API.

**Key Concept:** The function's **docstring** is critical. The ADK uses it as the tool's official description, which the LLM reads to understand its purpose, parameters, and when to use it.

In this example, we'll create a tool that calls the **free, public U.S. National Weather Service API** to get a real-time forecast. No API key needed!

In [ ]:
# --- Tool Definition: A function that calls a live public API ---
import requests
import json

# A simple lookup to avoid needing a separate geocoding API for this example
LOCATION_COORDINATES = {
    "sunnyvale": "37.3688,-122.0363",
    "san francisco": "37.7749,-122.4194",
    "lake tahoe": "39.0968,-120.0324"
}

def get_live_weather_forecast(location: str) -> dict:
    """Gets the current, real-time weather forecast for a specified location in the US.

    Args:
        location: The city name, e.g., "San Francisco".

    Returns:
        A dictionary containing the temperature and a detailed forecast.
    """
    print(f"🛠️ TOOL CALLED: get_live_weather_forecast(location='{location}')")

    # Find coordinates for the location
    normalized_location = location.lower()
    coords_str = None
    for key, val in LOCATION_COORDINATES.items():
        if key in normalized_location:
            coords_str = val
            break
    if not coords_str:
        return {"status": "error", "message": f"I don't have coordinates for {location}."}

    try:
        # NWS API requires 2 steps: 1. Get the forecast URL from the coordinates.
        points_url = f"https://api.weather.gov/points/{coords_str}"
        headers = {"User-Agent": "ADK Example Notebook"}
        points_response = requests.get(points_url, headers=headers)
        points_response.raise_for_status() # Raise an exception for bad status codes
        forecast_url = points_response.json()['properties']['forecast']

        # 2. Get the actual forecast from the URL.
        forecast_response = requests.get(forecast_url, headers=headers)
        forecast_response.raise_for_status()

        # Extract the relevant forecast details
        current_period = forecast_response.json()['properties']['periods'][0]
        return {
            "status": "success",
            "temperature": f"{current_period['temperature']}°{current_period['temperatureUnit']}",
            "forecast": current_period['detailedForecast']
        }
    except requests.exceptions.RequestException as e:
        return {"status": "error", "message": f"API request failed: {e}"}

# --- Agent Definition: An agent that USES the new tool ---

weather_agent = Agent(
    name="weather_aware_planner",
    model="gemini-2.5-flash",
    description="A trip planner that checks the real-time weather before making suggestions.",
    instruction="You are a cautious trip planner. Before suggesting any outdoor activities, you MUST use the `get_live_weather_forecast` tool to check conditions. Incorporate the live weather details into your recommendation.",
    tools=[get_live_weather_forecast]
)

print(f"🌦️ Agent '{weather_agent.name}' is created and can now call a live weather API!")

🌦️ Agent 'weather_aware_planner' is created and can now call a live weather API!


In [ ]:
# --- Let's test the Weather-Aware Planner ---

async def run_weather_planner_test():
    weather_session = await session_service.create_session(app_name=weather_agent.name, user_id=my_user_id)
    query = "I want to go hiking near Lake Tahoe, what's the weather like?"
    print(f"🗣️ User Query: '{query}'")
    await run_agent_query(weather_agent, query, weather_session, my_user_id)

await run_weather_planner_test()

🗣️ User Query: 'I want to go hiking near Lake Tahoe, what's the weather like?'

🚀 Running query for agent: 'weather_aware_planner' in session: '52da83bc-d922-4a87-b891-7e36596a56d2'...


EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'location': 'Lake Tahoe'
        },
        id='adk-101c37ea-932e-4aee-bac1-c9deb9d3aab5',
        name='get_live_weather_forecast'
      ),
      thought_signature=b'\n\xd2\x02\x01\x8f=k_\x9e\x96q\xad#\xd6GB\xc1M\xe4\xd3\xd2\x87\xcf\xfcz\xd0\x18\\\x0b4\x11\x15\x04G&\x0c\xe3\x16\xabR\xed\x01eD\xe0\xe6\x16\x91G\xac\x1a?9S\x967\xd3\x02\xaa\xcb\xc2\xd9\xe6\x02\xcc\x94Z9\xf8Hv\x0f\xe6\xb2\x89\x0f\xe65h\xc1\x15\xd6\xdb\x12Ka\xdd\x0e\xf3I\x19\xa7\x7f\x1d\x9a \xcb...'
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=10,
  candidates_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count

The weather near Lake Tahoe is not ideal for hiking today. There's a 70% chance of rain and snow showers, especially before 2 PM, with a slight chance of thunderstorms later. The temperature is around 44°F with a light west wind. I'd recommend choosing an indoor activity or postponing your hike for a day with better weather.

--------------------------------------------------



## 2.2 The Agent-as-a-Tool: Consulting a Specialist 🧑‍🍳

Why build one agent that does everything when you can build a **team of specialist agents?** The **Agent-as-a-Tool** pattern allows one agent to delegate a task to another agent.

**Key Concept:** This is different from a sub-agent. When Agent A calls Agent B as a tool, Agent B's response is passed **back to Agent A**. Agent A then uses that information to form its own final response to the user. It's a powerful way to compose complex behaviors from simpler, focused, and reusable agents.

### How It Works

Our top-level agent, the `trip_data_concierge_agent`, acts as the **Orchestrator**. It has two tools at its disposal:

1.  `call_db_agent`: A function that internally calls our `db_agent` to fetch raw data.
2.  `call_concierge_agent`: A function that calls the `concierge_agent`.

The `concierge_agent` itself has a tool: the `food_critic_agent`.

The flow for a complex query is:

1.  **User** asks the `trip_data_concierge_agent` for a hotel and a nearby restaurant.
2.  The **Orchestrator** first calls `call_db_agent` to get hotel data.
3.  The data is saved in `tool_context.state`.
4.  The **Orchestrator** then calls `call_concierge_agent`, which retrieves the hotel data from the context.
5.  The `concierge_agent` receives the request and decides it needs to use its own tool, the `food_critic_agent`.
6.  The `food_critic_agent` provides a witty recommendation.
7.  The `concierge_agent` gets the critic's response and politely formats it.
8.  This final, polished response is returned to the **Orchestrator**, which presents it to the user.

                         +-----------------------------------------------------------+
                         |              🧭 Trip Data Concierge Agent                 |
                         |-----------------------------------------------------------|
                         |  Model: gemini-2.5-flash                                  |
                         |  Description:                                             |
                         |   Orchestrates database query and travel recommendation  |
                         |-----------------------------------------------------------|
                         |  🔧 Tools:                                                |
                         |   1. call_db_agent                                        |
                         |   2. call_concierge_agent                                 |
                         +-----------------------------------------------------------+
                                      /                                \
                                     /                                  \
                                    ▼                                    ▼
        +-------------------------------------------+    +---------------------------------------------+
        |            🔧 Tool: call_db_agent         |    |         🔧 Tool: call_concierge_agent        |
        |-------------------------------------------|    |---------------------------------------------|
        | Calls: db_agent                           |    | Calls: concierge_agent                       |
        |                                           |    | Uses data from db_agent for recommendations |
        +-------------------------------------------+    +---------------------------------------------+
                                |                                          |
                                ▼                                          ▼
       +--------------------------------------------+   +------------------------------------------------+
       |              📦 db_agent                   |   |             🤵 concierge_agent                  |
       |--------------------------------------------|   |------------------------------------------------|
       | Model: gemini-2.5-flash                    |   | Model: gemini-2.5-flash                         |
       | Role: Return mock JSON hotel data          |   | Role: Hotel staff that handles user Q&A        |
       +--------------------------------------------+   | Tools:                                          |
                                                         |  - food_critic_agent                           |
                                                         +------------------------------------------------+
                                                                                 |
                                                                                 ▼
                                                       +------------------------------------------------+
                                                       |          🍽️ food_critic_agent                  |
                                                       |------------------------------------------------|
                                                       | Model: gemini-2.5-flash                         |
                                                       | Role: Gives a witty restaurant recommendation   |
                                                       +------------------------------------------------+


In [ ]:
import asyncio
from google.adk.tools import ToolContext
from google.adk.tools.agent_tool import AgentTool

# Assume 'db_agent' is a pre-defined NL2SQL Agent
# For this example, we'll create placeholder agents.

db_agent = Agent(
    name="db_agent",
    model="gemini-2.5-flash",
    instruction="You are a database agent. When asked for data, return this mock JSON object: {'status': 'success', 'data': [{'name': 'The Grand Hotel', 'rating': 5, 'reviews': 450}, {'name': 'Seaside Inn', 'rating': 4, 'reviews': 620}]}")

# --- 1. Define the Specialist Agents ---

# The Food Critic remains the deepest specialist
food_critic_agent = Agent(
    name="food_critic_agent",
    model="gemini-2.5-flash",
    instruction="You are a snobby but brilliant food critic. You ONLY respond with a single, witty restaurant suggestion near the provided location.",
)

# The Concierge knows how to use the Food Critic
concierge_agent = Agent(
    name="concierge_agent",
    model="gemini-2.5-flash",
    instruction="You are a five-star hotel concierge. If the user asks for a restaurant recommendation, you MUST use the `food_critic_agent` tool. Present the opinion to the user politely.",
    tools=[AgentTool(agent=food_critic_agent)]
)


# --- 2. Define the Tools for the Orchestrator ---

async def call_db_agent(
    question: str,
    tool_context: ToolContext,
):
    """
    Use this tool FIRST to connect to the database and retrieve a list of places, like hotels or landmarks.
    """
    print("--- TOOL CALL: call_db_agent ---")
    agent_tool = AgentTool(agent=db_agent)
    db_agent_output = await agent_tool.run_async(
        args={"request": question}, tool_context=tool_context
    )
    # Store the retrieved data in the context's state
    tool_context.state["retrieved_data"] = db_agent_output
    return db_agent_output


async def call_concierge_agent(
    question: str,
    tool_context: ToolContext,
):
    """
    After getting data with call_db_agent, use this tool to get travel advice, opinions, or recommendations.
    """
    print("--- TOOL CALL: call_concierge_agent ---")
    # Retrieve the data fetched by the previous tool
    input_data = tool_context.state.get("retrieved_data", "No data found.")

    # Formulate a new prompt for the concierge, giving it the data context
    question_with_data = f"""
    Context: The database returned the following data: {input_data}

    User's Request: {question}
    """

    agent_tool = AgentTool(agent=concierge_agent)
    concierge_output = await agent_tool.run_async(
        args={"request": question_with_data}, tool_context=tool_context
    )
    return concierge_output


# --- 3. Define the Top-Level Orchestrator Agent ---

trip_data_concierge_agent = Agent(
    name="trip_data_concierge",
    model="gemini-2.5-flash",
    description="Top-level agent that queries a database for travel data, then calls a concierge agent for recommendations.",
    tools=[call_db_agent, call_concierge_agent],
    instruction="""
    You are a master travel planner who uses data to make recommendations.

    1.  **ALWAYS start with the `call_db_agent` tool** to fetch a list of places (like hotels) that match the user's criteria.

    2.  After you have the data, **use the `call_concierge_agent` tool** to answer any follow-up questions for recommendations, opinions, or advice related to the data you just found.
    """,
)

print(f"✅ Orchestrator Agent '{trip_data_concierge_agent.name}' is defined and ready.")

✅ Orchestrator Agent 'trip_data_concierge' is defined and ready.


In [ ]:
# --- Let's test the Trip Data Concierge Agent ---

async def run_trip_data_concierge():
    """
    Sets up a session and runs a query against the top-level
    trip_data_concierge_agent.
    """
    # Create a new, single-use session for this query
    concierge_session = await session_service.create_session(
        app_name=trip_data_concierge_agent.name,
        user_id=my_user_id
    )

    # This query is specifically designed to trigger the full two-step process:
    # 1. Get data from the db_agent.
    # 2. Get a recommendation from the concierge_agent based on that data.
    query = "Find the top-rated hotels in San Francisco from the database, then suggest a dinner spot near the one with the most reviews."
    print(f"🗣️ User Query: '{query}'")

    # We call our existing helper function with the top-level orchestrator agent
    await run_agent_query(trip_data_concierge_agent, query, concierge_session, my_user_id)

# Run the test
await run_trip_data_concierge()

🗣️ User Query: 'Find the top-rated hotels in San Francisco from the database, then suggest a dinner spot near the one with the most reviews.'

🚀 Running query for agent: 'trip_data_concierge' in session: '57a1ba4c-e149-4efb-9fce-1a88a12f6ecc'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'question': 'top-rated hotels in San Francisco'
        },
        id='adk-41e1c664-6975-4d9c-8a0b-c55419c9f682',
        name='call_db_agent'
      ),
      thought_signature=b'\n\x8d\x05\x01\x8f=k_\x81Z\x14\xcaP\xa1\xca\n\xc3I0)g\xca\xfd\x0c\x1b$<\xe77i\'\xee\xcbL%j\x887\xcd~\xaf\xb1\xbaWW~\xfd\x86\x18\x0e"\x91\x01\xf0\x97\xf6I]\x1a<\xee\xb0\xc6\xe9\xac\xdc6\xfe\x16N!\xfd\x87?\xff\xa9j( I\x0e\x128qxZ\xceJ\xbb-\xacZ\xc0|O\x99l...'
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None cus

The top-rated hotels in San Francisco are The Grand Hotel (5 stars, 450 reviews) and Seaside Inn (4 stars, 620 reviews).

The Seaside Inn has the most reviews. For a dinner spot near the Seaside Inn, I suggest **Spruce**.

--------------------------------------------------



---
## Part 3: Agent with a Memory - The Adaptive Planner 🗺️

Now, let's see an agent that not only **remembers** but also **adapts**. We'll challenge the `multi_day_trip_agent` to re-plan part of its itinerary based on our feedback. This is a much more realistic test of conversational AI.

```
+-----------------------------------------------------+
|         Adaptive Multi-Day Trip Agent 🗺️           |
|-----------------------------------------------------|
|  Model: gemini-2.5-flash                            |
|  Description:                                       |
|   Builds multi-day travel itineraries step-by-step, |
|   remembers previous days, adapts to feedback       |
|-----------------------------------------------------|
|  🔧 Tools:                                          |
|   - Google Search                                   |
|-----------------------------------------------------|
|  🧠 Capabilities:                                   |
|   - Memory of past conversation & preferences       |
|   - Progressive planning (1 day at a time)          |
|   - Adapts to user feedback                         |
|   - Ensures activity variety across days            |
+-----------------------------------------------------+

            ▲
            |
    +---------------------------+
    |     User Interaction      |
    |---------------------------|
    | - Destination             |
    | - Trip duration           |
    | - Interests & feedback    |
    +---------------------------+

            |
            ▼

+-----------------------------------------------------+
|        Day-by-Day Itinerary Generation              |
|-----------------------------------------------------|
|  🗓️ Day N Output (Markdown format):                 |
|   - Morning / Afternoon / Evening activities        |
|   - Personalized & context-aware                    |
|   - Changes accepted, feedback acknowledged         |
+-----------------------------------------------------+

            |
            ▼

+-----------------------------------------------------+
|        Next Day Planning Triggered 🚀               |
|-----------------------------------------------------|
| - Builds on prior days                              |
| - Avoids repetition                                 |
| - Asks user for confirmation before proceeding      |
+-----------------------------------------------------+
```


In [ ]:
# --- Agent Definition: The Adaptive Planner ---

def create_multi_day_trip_agent():
    """Create the Progressive Multi-Day Trip Planner agent"""
    return Agent(
        name="multi_day_trip_agent",
        model="gemini-2.5-flash",
        description="Agent that progressively plans a multi-day trip, remembering previous days and adapting to user feedback.",
        instruction="""
        You are the "Adaptive Trip Planner" 🗺️ - an AI assistant that builds multi-day travel itineraries step-by-step.

        Your Defining Feature:
        You have short-term memory. You MUST refer back to our conversation to understand the trip's context, what has already been planned, and the user's preferences. If the user asks for a change, you must adapt the plan while keeping the unchanged parts consistent.

        Your Mission:
        1.  **Initiate**: Start by asking for the destination, trip duration, and interests.
        2.  **Plan Progressively**: Plan ONLY ONE DAY at a time. After presenting a plan, ask for confirmation.
        3.  **Handle Feedback**: If a user dislikes a suggestion (e.g., "I don't like museums"), acknowledge their feedback, and provide a *new, alternative* suggestion for that time slot that still fits the overall theme.
        4.  **Maintain Context**: For each new day, ensure the activities are unique and build logically on the previous days. Do not suggest the same things repeatedly.
        5.  **Final Output**: Return each day's itinerary in MARKDOWN format.
        """,
        tools=[google_search]
    )

multi_day_agent = create_multi_day_trip_agent()
print(f"🗺️ Agent '{multi_day_agent.name}' is created and ready to plan and adapt!")

🗺️ Agent 'multi_day_trip_agent' is created and ready to plan and adapt!


### Scenario 3a: Agent WITH Memory (Using a SINGLE Session) ✅

First, let's see the correct way to do it. We will use the **exact same `trip_session` object** for the entire conversation. Watch how the agent remembers the context from Turn 1 to correctly handle the requests in Turn 2 and 3.

In [ ]:
# --- Scenario 2: Testing Adaptation and Memory ---

async def run_adaptive_memory_demonstration():
    print("### 🧠 DEMO 2: AGENT THAT ADAPTS (SAME SESSION) ###")

    # Create ONE session that we will reuse for the whole conversation
    trip_session = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"Created a single session for our trip: {trip_session.id}")

    # --- Turn 1: The user initiates the trip ---
    query1 = "Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food."
    print(f"\n🗣️ User (Turn 1): '{query1}'")
    await run_agent_query(multi_day_agent, query1, trip_session, my_user_id)

    # --- Turn 2: The user gives FEEDBACK and asks for a CHANGE ---
    # We use the EXACT SAME `trip_session` object!
    query2 = "That sounds pretty good, but I'm not a huge fan of castles. Can you replace the morning activity for Day 1 with something else historical?"
    print(f"\n🗣️ User (Turn 2 - Feedback): '{query2}'")
    await run_agent_query(multi_day_agent, query2, trip_session, my_user_id)

    # --- Turn 3: The user confirms and asks to continue ---
    query3 = "Yes, the new plan for Day 1 is perfect! Please plan Day 2 now, keeping the food theme in mind."
    print(f"\n🗣️ User (Turn 3 - Confirmation): '{query3}'")
    await run_agent_query(multi_day_agent, query3, trip_session, my_user_id)

await run_adaptive_memory_demonstration()

### 🧠 DEMO 2: AGENT THAT ADAPTS (SAME SESSION) ###
Created a single session for our trip: ac7c9faf-ea1a-4f0d-a727-29a7ba9a0ad2

🗣️ User (Turn 1): 'Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: 'ac7c9faf-ea1a-4f0d-a727-29a7ba9a0ad2'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Great! Lisbon is a fantastic choice for a 2-day trip, especially with your interest in historic sites and local food. I'll help you plan a memorable itinerary.

Let's start with **Day 1**. How about we begin with some of Lisbon's most iconic historic landmarks and immerse ourselves in the local culinary scene?

Here's a possible plan for Day 1:

**Day 1: Alfama's Charms & Riverside Delights**

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama District & São Jorge Castle**
    *   Start your day in Alfama, Lisbon's oldest district, known for i

Great! Lisbon is a fantastic choice for a 2-day trip, especially with your interest in historic sites and local food. I'll help you plan a memorable itinerary.

Let's start with **Day 1**. How about we begin with some of Lisbon's most iconic historic landmarks and immerse ourselves in the local culinary scene?

Here's a possible plan for Day 1:

**Day 1: Alfama's Charms & Riverside Delights**

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama District & São Jorge Castle**
    *   Start your day in Alfama, Lisbon's oldest district, known for its narrow, winding streets, historic charm, and Fado music.
    *   Visit the imposing **São Jorge Castle (Castelo de São Jorge)**, offering panoramic views of the city and the Tagus River. Explore its ramparts, gardens, and archaeological site.
    *   Wander through the labyrinthine alleys of Alfama, discovering hidden viewpoints and traditional shops.
*   **Lunch (1:00 PM - 2:30 PM): Traditional Portuguese Cuisine in Alfama**
    *   Enjoy a traditional Portuguese lunch at a local tasca (tavern) in Alfama. Look for dishes like "Bacalhau à Brás" (codfish with scrambled eggs and potatoes) or grilled sardines.
*   **Afternoon (2:30 PM - 6:00 PM): Lisbon Cathedral & Baixa District**
    *   Walk to the **Lisbon Cathedral (Sé de Lisboa)**, the city's oldest church, and admire its Romanesque architecture.
    *   Continue your stroll down to the **Baixa District**, rebuilt after the 1755 earthquake with a grid-like street plan. Explore Praça do Comércio, one of Europe's largest and most beautiful squares, right on the riverfront.
*   **Evening (6:00 PM onwards): Sunset Views & Dinner in Chiado/Bairro Alto**
    *   Take the Santa Justa Lift (Elevador de Santa Justa) or walk up to the **Chiado** or **Bairro Alto** districts.
    *   Enjoy a breathtaking sunset view from a *miradouro* (viewpoint) like Miradouro de São Pedro de Alcântara.
    *   For dinner, explore the vibrant restaurant scene in Chiado or Bairro Alto, offering a wide range of local and international culinary options. Perhaps try some *petiscos* (Portuguese tapas).

How does this sound for your first day in Lisbon? Would you like any adjustments, or shall we move on to Day 2?

--------------------------------------------------


🗣️ User (Turn 2 - Feedback): 'That sounds pretty good, but I'm not a huge fan of castles. Can you replace the morning activity for Day 1 with something else historical?'

🚀 Running query for agent: 'multi_day_trip_agent' in session: 'ac7c9faf-ea1a-4f0d-a727-29a7ba9a0ad2'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Okay, no problem at all! I can definitely swap out the castle for another historical site that fits your interests. The Alfama district itself is a treasure trove of history.

Here's an updated plan for Day 1, replacing the castle with another significant historical landmark:

**Day 1: Alfama's Charms & Riverside Delights (Revised)**

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama District & National Pantheon**
    *   Start your day in Alfama, Lisbon's oldest district, known for its narrow, winding streets, historic charm, and Fado music.
    *   Visit the **National Pan

Okay, no problem at all! I can definitely swap out the castle for another historical site that fits your interests. The Alfama district itself is a treasure trove of history.

Here's an updated plan for Day 1, replacing the castle with another significant historical landmark:

**Day 1: Alfama's Charms & Riverside Delights (Revised)**

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama District & National Pantheon**
    *   Start your day in Alfama, Lisbon's oldest district, known for its narrow, winding streets, historic charm, and Fado music.
    *   Visit the **National Pantheon (Panteão Nacional)**, a magnificent baroque building that serves as the resting place for many important Portuguese personalities. You can explore its grand interior and enjoy panoramic views of the city from its dome.
    *   Wander through the labyrinthine alleys of Alfama, discovering hidden viewpoints and traditional shops.
*   **Lunch (1:00 PM - 2:30 PM): Traditional Portuguese Cuisine in Alfama**
    *   Enjoy a traditional Portuguese lunch at a local tasca (tavern) in Alfama. Look for dishes like "Bacalhau à Brás" (codfish with scrambled eggs and potatoes) or grilled sardines.
*   **Afternoon (2:30 PM - 6:00 PM): Lisbon Cathedral & Baixa District**
    *   Walk to the **Lisbon Cathedral (Sé de Lisboa)**, the city's oldest church, and admire its Romanesque architecture.
    *   Continue your stroll down to the **Baixa District**, rebuilt after the 1755 earthquake with a grid-like street plan. Explore Praça do Comércio, one of Europe's largest and most beautiful squares, right on the riverfront.
*   **Evening (6:00 PM onwards): Sunset Views & Dinner in Chiado/Bairro Alto**
    *   Take the Santa Justa Lift (Elevador de Santa Justa) or walk up to the **Chiado** or **Bairro Alto** districts.
    *   Enjoy a breathtaking sunset view from a *miradouro* (viewpoint) like Miradouro de São Pedro de Alcântara.
    *   For dinner, explore the vibrant restaurant scene in Chiado or Bairro Alto, offering a wide range of local and international culinary options. Perhaps try some *petiscos* (Portuguese tapas).

How does this revised Day 1 plan look to you? Does the National Pantheon better suit your preference for historical sites?

--------------------------------------------------


🗣️ User (Turn 3 - Confirmation): 'Yes, the new plan for Day 1 is perfect! Please plan Day 2 now, keeping the food theme in mind.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: 'ac7c9faf-ea1a-4f0d-a727-29a7ba9a0ad2'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Excellent! I'm glad Day 1 is perfect. Let's dive into **Day 2**, focusing on more historical discoveries and, of course, some incredible local food experiences.

For your second day, I recommend exploring the historic Belém district, famous for its monumental architecture linked to the Age of Discoveries, and then immersing ourselves in Lisbon's vibrant food scene.

Here's a possible plan for Day 2:

**Day 2: Age of Discoveries & Culinary Immersion**

*   **Morning (9:00 AM - 1:00 PM): Belém District Exploration**
    *   Begin your day in **Belém**, a district of immense historical significance, particularly duri

Excellent! I'm glad Day 1 is perfect. Let's dive into **Day 2**, focusing on more historical discoveries and, of course, some incredible local food experiences.

For your second day, I recommend exploring the historic Belém district, famous for its monumental architecture linked to the Age of Discoveries, and then immersing ourselves in Lisbon's vibrant food scene.

Here's a possible plan for Day 2:

**Day 2: Age of Discoveries & Culinary Immersion**

*   **Morning (9:00 AM - 1:00 PM): Belém District Exploration**
    *   Begin your day in **Belém**, a district of immense historical significance, particularly during Portugal's Age of Discoveries.
    *   Visit the magnificent **Jerónimos Monastery (Mosteiro dos Jerónimos)**, a UNESCO World Heritage site and a breathtaking example of Manueline architecture. Explore its ornate church and tranquil cloisters.
    *   Stroll along the Tagus River to see two other iconic landmarks: the **Belém Tower (Torre de Belém)**, a fortified tower that once guarded the entrance to the city's harbor, and the **Monument to the Discoveries (Padrão dos Descobrimentos)**, celebrating the Portuguese explorers.
*   **Lunch & Sweet Treat (1:00 PM - 2:30 PM): Iconic Pastéis de Belém & Local Flavors**
    *   No visit to Belém is complete without trying the original and world-famous **Pastéis de Belém** at the historic factory where they've been made since 1837. Enjoy them warm with a sprinkle of cinnamon!
    *   For lunch, find a local spot in Belém for an authentic Portuguese meal. Perhaps try a "bifana" (succulent pork sandwich) or a fresh seafood dish.
*   **Afternoon (2:30 PM - 5:30 PM): Carmo Convent & Rossio Square**
    *   Head back towards the city center and explore the atmospheric ruins of the **Carmo Convent (Convento do Carmo)**. This former church, largely destroyed by the 1755 earthquake, now stands as an open-air archaeological museum, offering a poignant glimpse into Lisbon's past.
    *   Wander through **Rossio Square (Praça do Rossio)**, a lively central square that has been a main focal point of Lisbon for centuries, surrounded by beautiful 18th-century buildings and fountains.
*   **Late Afternoon/Early Evening (5:30 PM - 7:00 PM): Time Out Market (Mercado da Ribeira)**
    *   Immerse yourself in Lisbon's modern culinary scene at the **Time Out Market (Mercado da Ribeira)**. This historic market building now houses a vibrant food hall featuring stalls from many of Lisbon's top chefs and traditional eateries. It's a fantastic place to sample a variety of local specialties and international cuisine.
*   **Evening (7:00 PM onwards): Farewell Dinner & Fado Experience**
    *   For your farewell dinner, consider a restaurant in the **Bairro Alto** or **Alfama** district, perhaps one that offers a traditional **Fado show**. This iconic Portuguese musical genre, recognized by UNESCO, provides a deeply emotional and authentic cultural experience to round off your trip.

How does this plan for Day 2 sound to you? We can certainly make adjustments if there's anything you'd like to change!

--------------------------------------------------



### Scenario 3b: Agent WITHOUT Memory (Using SEPARATE Sessions) ❌

Now, let's see what happens if we mess up our session management. Here, we'll give the agent a case of amnesia by creating a **brand new, separate session for each turn**.

Pay close attention to the agent's response to the second query. Because it's in a new session, it has no memory of the trip to Lisbon we just discussed!

In [ ]:
# --- Scenario 2b: Demonstrating Memory FAILURE ---

async def run_memory_failure_demonstration():
    print("\n" + "#"*60)
    print("### 🧠 DEMO 2b: AGENT WITH AMNESIA (SEPARATE SESSIONS) ###")
    print("#"*60)

    # --- Turn 1: The user initiates the trip in the FIRST session ---
    query1 = "Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food."
    session_one = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"\nCreated a session for Turn 1: {session_one.id}")
    print(f"🗣️ User (Turn 1): '{query1}'")
    await run_agent_query(multi_day_agent, query1, session_one, my_user_id)

    # --- Turn 2: The user asks to continue... but in a completely NEW session ---
    query2 = "Yes, that looks perfect! Please plan Day 2."
    session_two = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"\nCreated a BRAND NEW session for Turn 2: {session_two.id}")
    print(f"🗣️ User (Turn 2): '{query2}'")
    await run_agent_query(multi_day_agent, query2, session_two, my_user_id)

await run_memory_failure_demonstration()


############################################################
### 🧠 DEMO 2b: AGENT WITH AMNESIA (SEPARATE SESSIONS) ###
############################################################

Created a session for Turn 1: 166d7804-c59c-4e7b-9de5-0053ec439823
🗣️ User (Turn 1): 'Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '166d7804-c59c-4e7b-9de5-0053ec439823'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Hello! A 2-day trip to Lisbon sounds wonderful, especially with your interest in historic sites and local food.

Let's plan out your first day. How about this for Day 1?

**Day 1: Discovering Belém's Maritime History and Alfama's Charms**

*   **Morning (9:00 AM - 1:00 PM): Explore Belém's Historic Gems**
    Start your day in the historic district of Belém, renowned for its monuments celebrating Portugal's Age of Discoveri

Hello! A 2-day trip to Lisbon sounds wonderful, especially with your interest in historic sites and local food.

Let's plan out your first day. How about this for Day 1?

**Day 1: Discovering Belém's Maritime History and Alfama's Charms**

*   **Morning (9:00 AM - 1:00 PM): Explore Belém's Historic Gems**
    Start your day in the historic district of Belém, renowned for its monuments celebrating Portugal's Age of Discoveries.
    *   Visit the iconic **Belém Tower (Torre de Belém)**, a UNESCO World Heritage site that once guarded the entrance to Lisbon's harbor.
    *   Next, explore the magnificent **Jerónimos Monastery (Mosteiro dos Jerónimos)**, another UNESCO World Heritage site, known for its stunning Manueline architecture and the tomb of Vasco da Gama.
*   **Lunch (1:00 PM - 2:30 PM): Taste the Original Pastel de Nata**
    Enjoy lunch in Belém. Afterward, no visit to Belém is complete without trying the original and famous **Pastéis de Belém** at the historic factory.
*   **Afternoon (2:30 PM - 6:00 PM): Wander Through Alfama and Lisbon Cathedral**
    Head back towards the city center to Alfama, Lisbon's oldest district.
    *   Get lost in the labyrinthine streets of **Alfama**, soaking in its medieval atmosphere, Fado music, and picturesque alleys.
    *   Visit the **Lisbon Cathedral (Sé de Lisboa)**, the city's oldest church, showcasing a mix of architectural styles from Romanesque to Gothic.
*   **Evening (7:30 PM onwards): Traditional Portuguese Dinner with Fado**
    For dinner, enjoy traditional Portuguese cuisine in Alfama or the nearby Baixa district, perhaps at a restaurant offering live Fado music for an authentic cultural experience.

How does this sound for your first day in Lisbon?

--------------------------------------------------


Created a BRAND NEW session for Turn 2: 0dd38a57-9f30-4b58-b64d-12f1a9b69607
🗣️ User (Turn 2): 'Yes, that looks perfect! Please plan Day 2.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '0dd38a57-9f30-4b58-b64d-12f1a9b69607'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Okay, fantastic! Let's plan out your second day in London, keeping in mind your interests in history, art, and good food.

Here's a proposed itinerary for Day 2:

### **Day 2: Art, Culture & West End Charm**

*   **Morning (9:00 AM - 1:00 PM): The British Museum**
    Begin your day with a deep dive into human history, art, and culture at the world-renowned British Museum. Explore vast collections ranging from ancient Egyptian mummies to the Rosetta Stone and the Elgin Marbles. Allow ample time, as it's easy to spend hours here.
*   **Lunch (1:00 PM - 2:00 PM): Bloomsbury Delights**
    Enjoy a well-des

Okay, fantastic! Let's plan out your second day in London, keeping in mind your interests in history, art, and good food.

Here's a proposed itinerary for Day 2:

### **Day 2: Art, Culture & West End Charm**

*   **Morning (9:00 AM - 1:00 PM): The British Museum**
    Begin your day with a deep dive into human history, art, and culture at the world-renowned British Museum. Explore vast collections ranging from ancient Egyptian mummies to the Rosetta Stone and the Elgin Marbles. Allow ample time, as it's easy to spend hours here.
*   **Lunch (1:00 PM - 2:00 PM): Bloomsbury Delights**
    Enjoy a well-deserved lunch at one of the many cafes within the British Museum, or explore the charming Bloomsbury neighborhood surrounding it, which offers a variety of independent eateries and pubs.
*   **Afternoon (2:00 PM - 5:30 PM): Covent Garden Exploration**
    Take a leisurely stroll or a short tube ride to Covent Garden. Immerse yourself in the vibrant atmosphere, browse the unique stalls in the Apple Market and Jubilee Market, enjoy the street performers in the piazza, and explore the boutiques and shops in the surrounding area.
*   **Evening (6:00 PM onwards): West End Show & Dinner**
    Savor an early dinner in the Covent Garden area, which boasts an array of dining options from casual to fine dining. Afterwards, catch a spectacular West End theatre show. London's Theatreland offers a diverse selection of musicals, plays, and performances to suit every taste.

How does this sound for your second day?

--------------------------------------------------



See? The agent was confused! It likely asked what destination or what trip we were talking about. Because the second query was in a fresh, isolated session, the agent had no memory of planning Day 1 in Lisbon.

This perfectly illustrates why **managing sessions is the key to building truly conversational agents!**

---
## 🎉 Congratulations! 🎉

Congratulations on completing your ADK adventure into Tools and Memory! You've taken a massive leap from building single-shot agents to creating dynamic, stateful AI systems.

Let's recap the powerful concepts you've mastered:

- **Fundamental Agent & Tools**: You started by building a "Day Trip Genie" and equipped it with its first tool, GoogleSearch.

- **Custom Function Tools**: You gave your agent a new sense by creating a custom tool to fetch live data from the U.S. National Weather Service API.

- **Agent-as-a-Tool**: You orchestrated a sophisticated hierarchy where agents delegate tasks to other, more specialized agents, creating a collaborative team.

- **The Power of Memory**: Most importantly, you saw firsthand how managing a single, persistent Session allows an agent to remember context, adapt to user feedback, and conduct a meaningful, multi-turn conversation.

```
   __            /\_/\         /\_/\        /\_/\         __             (\__/)
o-''|\_____/).  ( o.o )       ( -.- )      ( ^_^ )     o-''|\_____/).    ( ^_^ )
 \_/|_)     )    > ^ <         > * <        >💖<         \_/|_)     )     / >🌸< \
    \  __  /                                              \  __  /         /   \
    (_/ (_/                                               (_/ (_/        (___|___)
```
